# 附录A：必要数学基础

In [1]:
from pathlib import Path
import sys
root = Path.cwd().resolve()
while not (root / 'lsq').is_dir():
    if root == root.parent:
        raise RuntimeError('Run inside the repository')
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import least_squares
from lsq import (finite_difference, whiten, gauss_newton,
                 levenberg_marquardt, dogleg, recursive_update)
np.set_printoptions(precision=5, suppress=True)
plt.rcParams.update({'figure.figsize': (6.4, 3.3), 'font.size': 10,
                     'axes.grid': True, 'savefig.dpi': 130})

本附录按“在正文哪里会用到”组织。第一次阅读不必记住所有名词，能够识别维度、检查方向和解释条件即可。

## 向量、内积与范数

向量$x\in\mathbb R^n$有$n$个分量，内积$x^Ty=\sum_jx_jy_j$衡量方向关系。$\|x\|_2=\sqrt{x^Tx}$是欧氏长度，$\|x\|_\infty=\max_j|x_j|$用于逐分量停止判断。$x^Ty=0$表示正交，只有两个向量都非零时，才可以进一步定义几何夹角。

矩阵$A\in\mathbb R^{m\times n}$把$n$维参数映射到$m$维观测；$A^T$交换行列。$AB$可乘要求$A$列数等于$B$行数。编程中的一维数组没有显式行列方向，应根据运算结果的shape检查含义。

## 导数、梯度与二阶展开

标量目标$F$的梯度是所有一阶偏导组成的列向量。向量残差$r$的导数是雅可比$J$。链式法则给出$\nabla(\tfrac12r^Tr)=J^Tr$。足够光滑时

$$F(x+h)=F(x)+g^Th+\frac12h^THh+O(\|h\|^3).$$

三阶余项写法需要相应光滑性条件。若只假设二阶连续可微，更一般地写余项$o(\|h\|^2)$。

## 正定性、特征值与奇异值

实对称矩阵$B$正定，指所有非零$x$满足$x^TBx>0$，等价于所有特征值为正。此时二次目标有唯一极小点，可作Cholesky分解。半正定允许零方向，不保证唯一性。$J^TJ$总半正定，$J$满列秩时才正定。

一般矩阵的奇异值非负，平方等于$A^TA$的特征值。最大/最小非零奇异值之比描述可辨子空间上的敏感性；秩亏时通常说整个逆问题的条件数无穷大，不能把有限非零比值当作全参数可逆性。

In [2]:
A=np.array([[1.,0.],[1.,1.],[1.,2.]])
singular=np.linalg.svd(A,compute_uv=False)
eigen=np.linalg.eigvalsh(A.T@A)
print('singular values squared:',singular**2)
print('eigenvalues (ascending):',eigen)
assert np.allclose(np.sort(singular**2),eigen)

singular values squared: [7.16228 0.83772]
eigenvalues (ascending): [0.83772 7.16228]


## 概率与协方差

期望描述重复实验的平均位置，协方差$C=\mathbb E[(x-\mathbb Ex)(x-\mathbb Ex)^T]$描述波动及相关方向。线性变换$z=Bx$后，$\operatorname{Cov}(z)=BCB^T$。不相关仅指协方差为零，一般不等于独立；联合高斯等附加条件下才有更强结论。

## 练习与解答

练习：$A$为$20\times3$，$r$为20维，$J^Tr$是什么维度？解答：3维，与参数维度一致。

练习：$J$秩亏时，加$\mu I$的$J^TJ+\mu I$是否正定？解答：$\mu>0$时是；对非零$h$，$h^T(J^TJ+\mu I)h=\|Jh\|^2+\mu\|h\|^2>0$。这改善局部步求解，但没有让原数据突然能够辨识所有参数。